# RAG Assignment: Baseline Retrieval vs. Cross-Encoder Re-Ranking

## 1. Environment Setup and Configuration

This notebook compares two Retrieval-Augmented Generation (RAG) pipelines using a constitutional law PDF.

- **Pipeline 1 — Baseline:** Retrieve the Top 10 chunks using dense vector similarity and select the Top 3.
- **Pipeline 2 — Re-ranking:** Retrieve the same Top 10 chunks, re-rank them using a Cross-Encoder, and select the new Top 3.

Both pipelines use the same PDF, queries, and answer-generation prompt. We compare their retrieval rankings, generated answers, and execution times.

In [47]:
from pathlib import Path
import sys
import torch
import pymupdf
import faiss
import pandas as pd
import numpy as np

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"

PDF_FILES = list(DATA_DIR.glob("*.pdf"))

print("Python executable:", sys.executable)
print("Project directory:", PROJECT_DIR)
print("Device: CPU")
print("CUDA available:", torch.cuda.is_available())
print("PDF files found:", [p.name for p in PDF_FILES])

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert len(PDF_FILES) > 0, (
    "No PDF found. Place your constitutional law PDF inside the data folder."
)

PDF_PATH = PDF_FILES[0]
print("\nSelected PDF:", PDF_PATH.name)
print("Environment setup completed.")


Python executable: c:\Users\HP\Downloads\LMA_Assignment1\lma_env\Scripts\python.exe
Project directory: c:\Users\HP\Downloads\LMA_Assignment1
Device: CPU
CUDA available: False
PDF files found: ['OJ_L_202401689_EN_TXT.pdf']

Selected PDF: OJ_L_202401689_EN_TXT.pdf
Environment setup completed.


## 2. PDF Loading and Text Extraction

This section extracts text from each page of the EU AI Act PDF. We preserve page numbers so that retrieved chunks can be traced back to their source pages.

In [48]:
import pymupdf

with pymupdf.open(PDF_PATH) as pdf:
    for i in [0, 1, 2]:
        page = pdf[i]
        text = page.get_text("text").strip()

        print(f"\nPage {i + 1}")
        print("Extracted characters:", len(text))
        print("Embedded images:", len(page.get_images(full=True)))
        print("Text preview:", repr(text[:300]))



Page 1
Extracted characters: 3599
Embedded images: 1
Text preview: 'REGULATION (EU) 2024/1689 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL\nof 13 June 2024\nlaying down harmonised rules on artificial intelligence and amending Regulations (EC) No 300/2008, \n(EU) No 167/2013, (EU) No 168/2013, (EU) 2018/858, (EU) 2018/1139 and (EU) 2019/2144 and \nDirectives 2014/90/EU,'

Page 2
Extracted characters: 5130
Embedded images: 0
Text preview: 'guaranteeing the uniform protection of overriding reasons of public interest and of rights of persons throughout the \ninternal market on the basis of Article 114 of the Treaty on the Functioning of the European Union (TFEU). To the \nextent that this Regulation contains specific rules on the protecti'

Page 3
Extracted characters: 6884
Embedded images: 0
Text preview: '(9)\nHarmonised rules applicable to the placing on the market, the putting into service and the use of high-risk AI \nsystems should be laid down consistently with Regulation (EC) No 

In [49]:
pages = []

with pymupdf.open(PDF_PATH) as pdf:
    total_pdf_pages = len(pdf)

    for page_num, page in enumerate(pdf, start=1):
        text = page.get_text("text").strip()

        if text:
            pages.append({
                "page_number": page_num,
                "text": text
            })

print("Total PDF pages:", total_pdf_pages)
print("Pages with extractable text:", len(pages))
print("Total extracted characters:", sum(len(p["text"]) for p in pages))

print("\nFirst page preview:\n")
print(pages[0]["text"][:1000] if pages else "No text extracted.")


Total PDF pages: 144
Pages with extractable text: 144
Total extracted characters: 599631

First page preview:

REGULATION (EU) 2024/1689 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL
of 13 June 2024
laying down harmonised rules on artificial intelligence and amending Regulations (EC) No 300/2008, 
(EU) No 167/2013, (EU) No 168/2013, (EU) 2018/858, (EU) 2018/1139 and (EU) 2019/2144 and 
Directives 2014/90/EU, (EU) 2016/797 and (EU) 2020/1828 (Artificial Intelligence Act)
(Text with EEA relevance)
THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION,
Having regard to the Treaty on the Functioning of the European Union, and in particular Articles 16 and 114 thereof,
Having regard to the proposal from the European Commission,
After transmission of the draft legislative act to the national parliaments,
Having regard to the opinion of the European Economic and Social Committee (1),
Having regard to the opinion of the European Central Bank (2),
Having regard to the opinion of the Com

### 2.1. Extracted Text Quality Assessment

Before building the retrieval index, inspect the extracted pages for empty content, unusually short pages, repeated headers, and formatting artifacts. Since this is a legal document, preserve article numbers, section headings, exceptions, and references.

In [50]:
import pandas as pd
import re

page_df = pd.DataFrame(pages)

page_df["character_count"] = page_df["text"].str.len()
page_df["word_count"] = page_df["text"].str.split().str.len()

print("Total pages:", len(page_df))
print("Empty pages:", (page_df["character_count"] == 0).sum())
print("\nCharacter count summary:")
print(page_df["character_count"].describe())
print("\nWord count summary:")
print(page_df["word_count"].describe())

print("\nShortest 10 pages:")
print(
    page_df[["page_number", "character_count", "word_count"]]
    .sort_values("character_count")
    .head(10)
    .to_string(index=False)
)


Total pages: 144
Empty pages: 0

Character count summary:
count     144.000000
mean     4164.104167
std      1330.156690
min       254.000000
25%      3486.250000
50%      4058.000000
75%      5327.750000
max      6884.000000
Name: character_count, dtype: float64

Word count summary:
count     144.000000
mean      634.868056
std       199.779187
min        33.000000
25%       528.500000
50%       623.500000
75%       797.250000
max      1065.000000
Name: word_count, dtype: float64

Shortest 10 pages:
 page_number  character_count  word_count
         142              254          33
         129              493          76
         133              807         120
         137              936         150
         126              953         137
         138              979         154
         140             1384         198
         132             1504         237
         144             1587         249
         143             1617         254


### 2.2. PDF Text Preprocessing

PDF extraction can introduce formatting artifacts, including standalone page numbers, repeated headers and footers, excessive whitespace, and words split across lines. This section cleans formatting while preserving legal provisions, article numbers, paragraph numbering, and cross-references.

In [51]:
for page in pages[:5]:
    lines = page["text"].splitlines()

    print(f"\n--- Page {page['page_number']} ---")
    print("First 5 lines:", lines[:5])
    print("Last 5 lines:", lines[-5:])



--- Page 1 ---
First 5 lines: ['REGULATION (EU) 2024/1689 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL', 'of 13 June 2024', 'laying down harmonised rules on artificial intelligence and amending Regulations (EC) No 300/2008, ', '(EU) No 167/2013, (EU) No 168/2013, (EU) 2018/858, (EU) 2018/1139 and (EU) 2019/2144 and ', 'Directives 2014/90/EU, (EU) 2016/797 and (EU) 2020/1828 (Artificial Intelligence Act)']
Last 5 lines: ['(3)', 'OJ C 97, 28.2.2022, p. 60.', '(4)', 'Position of the European Parliament of 13 March 2024 (not yet published in the Official Journal) and decision of the Council of ', '21 May 2024.']

--- Page 2 ---
First 5 lines: ['guaranteeing the uniform protection of overriding reasons of public interest and of rights of persons throughout the ', 'internal market on the basis of Article 114 of the Treaty on the Functioning of the European Union (TFEU). To the ', 'extent that this Regulation contains specific rules on the protection of individuals with regard to the proces

In [52]:
import re

def clean_page_text(text):
    cleaned_lines = []

    for line in text.splitlines():
        line = line.strip()

        if not line:
            continue

        # Remove standalone page numbers, not numbered legal clauses.
        if re.fullmatch(r"\d{1,3}", line):
            continue

        cleaned_lines.append(line)

    text = "\n".join(cleaned_lines)
    text = text.replace("\u00ad", "")  # Soft hyphen
    text = text.replace("\x00", "")    # Null character
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


cleaned_pages = [
    {
        "page_number": page["page_number"],
        "text": clean_page_text(page["text"])
    }
    for page in pages
]

print("Pages before cleaning:", len(pages))
print("Pages after cleaning:", len(cleaned_pages))
print("Characters before:", sum(len(p["text"]) for p in pages))
print("Characters after:", sum(len(p["text"]) for p in cleaned_pages))


Pages before cleaning: 144
Pages after cleaning: 144
Characters before: 599631
Characters after: 595562


In [53]:

for page in cleaned_pages[:2]:
    print(f"\n--- Cleaned Page {page['page_number']} ---")
    print(page["text"][:1200])



--- Cleaned Page 1 ---
REGULATION (EU) 2024/1689 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL
of 13 June 2024
laying down harmonised rules on artificial intelligence and amending Regulations (EC) No 300/2008,
(EU) No 167/2013, (EU) No 168/2013, (EU) 2018/858, (EU) 2018/1139 and (EU) 2019/2144 and
Directives 2014/90/EU, (EU) 2016/797 and (EU) 2020/1828 (Artificial Intelligence Act)
(Text with EEA relevance)
THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION,
Having regard to the Treaty on the Functioning of the European Union, and in particular Articles 16 and 114 thereof,
Having regard to the proposal from the European Commission,
After transmission of the draft legislative act to the national parliaments,
Having regard to the opinion of the European Economic and Social Committee (1),
Having regard to the opinion of the European Central Bank (2),
Having regard to the opinion of the Committee of the Regions (3),
Acting in accordance with the ordinary legislative procedure (

## 3. Text Chunking and Metadata

The extracted document is divided into overlapping chunks. Each chunk retains its unique ID and source page number, allowing retrieved passages to be traced back to the original PDF.

In [54]:
def make_chunks(pages, chunk_size=450, overlap=80):
    chunks = []
    chunk_id = 0

    for page in pages:
        words = page["text"].split()
        start = 0

        while start < len(words):
            end = min(start + chunk_size, len(words))

            chunks.append({
                "chunk_id": chunk_id,
                "page_number": page["page_number"],
                "text": " ".join(words[start:end])
            })

            chunk_id += 1

            if end == len(words):
                break

            start = end - overlap

    return chunks


chunks = make_chunks(cleaned_pages)

print("Total chunks:", len(chunks))
print("Unique chunk IDs:", len(set(c["chunk_id"] for c in chunks)))
print("Empty chunks:", sum(not c["text"].strip() for c in chunks))
print("Pages represented:", len(set(c["page_number"] for c in chunks)))

print("\nFirst chunk:\n")
print(chunks[0]["text"][:1000])


Total chunks: 300
Unique chunk IDs: 300
Empty chunks: 0
Pages represented: 144

First chunk:

REGULATION (EU) 2024/1689 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL of 13 June 2024 laying down harmonised rules on artificial intelligence and amending Regulations (EC) No 300/2008, (EU) No 167/2013, (EU) No 168/2013, (EU) 2018/858, (EU) 2018/1139 and (EU) 2019/2144 and Directives 2014/90/EU, (EU) 2016/797 and (EU) 2020/1828 (Artificial Intelligence Act) (Text with EEA relevance) THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty on the Functioning of the European Union, and in particular Articles 16 and 114 thereof, Having regard to the proposal from the European Commission, After transmission of the draft legislative act to the national parliaments, Having regard to the opinion of the European Economic and Social Committee (1), Having regard to the opinion of the European Central Bank (2), Having regard to the opinion of the Committee of the Regio

## 4. Baseline Retrieval: Bi-Encoder and FAISS

The `BAAI/bge-base-en-v1.5` embedding model encodes each chunk independently into a dense vector. We normalize the vectors and use FAISS inner-product search, which is equivalent to cosine similarity for normalized vectors.

In [55]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

DEVICE = "cpu"
TOP_K = 10
FINAL_K = 3

embedding_model = SentenceTransformer(
    "BAAI/bge-base-en-v1.5",
    device=DEVICE
)

print("Embedding model loaded:", "BAAI/bge-base-en-v1.5")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model loaded: BAAI/bge-base-en-v1.5


In [56]:
chunk_texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=16,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print("Embedding matrix shape:", embeddings.shape)
print("Vectors indexed:", index.ntotal)

print("Vector dimensions:", embeddings.shape[1])


Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Embedding matrix shape: (300, 768)
Vectors indexed: 300
Vector dimensions: 768


In [57]:
def retrieve_top10(query):
    query_text = (
        "Represent this sentence for searching relevant passages: " + query
    )

    query_embedding = embedding_model.encode(
        [query_text],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(query_embedding, TOP_K)

    results = []

    for rank, (idx, score) in enumerate(
        zip(indices[0], scores[0]), start=1
    ):
        if idx < 0:
            continue

        results.append({
            **chunks[int(idx)],
            "initial_rank": rank,
            "bi_score": float(score)
        })

    return results


In [58]:
query = "What obligations apply to providers of high-risk AI systems?"

baseline_results = retrieve_top10(query)

for item in baseline_results:
    print(
        f"Rank: {item['initial_rank']} | "
        f"Page: {item['page_number']} | "
        f"Score: {item['bi_score']:.4f}"
    )
    print(item["text"][:350])
    print("-" * 80)

Rank: 1 | Page: 62 | Score: 0.8099
SECTION 3 Obligations of providers and deployers of high-risk AI systems and other parties Article 16 Obligations of providers of high-risk AI systems Providers of high-risk AI systems shall: (a) ensure that their high-risk AI systems are compliant with the requirements set out in Section 2; (b) indicate on the high-risk AI system or, where that is
--------------------------------------------------------------------------------
Rank: 2 | Page: 67 | Score: 0.7815
Article 25 Responsibilities along the AI value chain 1. Any distributor, importer, deployer or other third-party shall be considered to be a provider of a high-risk AI system for the purposes of this Regulation and shall be subject to the obligations of the provider under Article 16, in any of the following circumstances: (a) they put their name or
--------------------------------------------------------------------------------
Rank: 3 | Page: 56 | Score: 0.7579
2. Where a product contains an 

In [59]:
baseline_top3 = baseline_results[:FINAL_K]

for item in baseline_top3:
    print(f"Rank {item['initial_rank']} | Page {item['page_number']}")
    print(item["text"][:500])
    print("-" * 80)

Rank 1 | Page 62
SECTION 3 Obligations of providers and deployers of high-risk AI systems and other parties Article 16 Obligations of providers of high-risk AI systems Providers of high-risk AI systems shall: (a) ensure that their high-risk AI systems are compliant with the requirements set out in Section 2; (b) indicate on the high-risk AI system or, where that is not possible, on its packaging or its accompanying documentation, as applicable, their name, registered trade name or registered trade mark, the addr
--------------------------------------------------------------------------------
Rank 2 | Page 67
Article 25 Responsibilities along the AI value chain 1. Any distributor, importer, deployer or other third-party shall be considered to be a provider of a high-risk AI system for the purposes of this Regulation and shall be subject to the obligations of the provider under Article 16, in any of the following circumstances: (a) they put their name or trademark on a high-risk AI syste

## Section 5 — Cross-Encoder Reranking

The Bi-Encoder compares the question and document chunks through their separately generated embeddings. The Cross-Encoder instead processes the question and each candidate chunk together, allowing it to assess their relevance more directly.

In [60]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L6-v2",
    device=DEVICE
)

print("Cross-Encoder loaded successfully.")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Cross-Encoder loaded successfully.


In [61]:
def rerank_top10(query, candidates):
    pairs = [(query, candidate["text"]) for candidate in candidates]

    scores = reranker.predict(pairs)

    reranked = []

    for candidate, score in zip(candidates, scores):
        reranked.append({
            **candidate,
            "cross_score": float(score)
        })

    reranked.sort(
        key=lambda item: item["cross_score"],
        reverse=True
    )

    for rank, candidate in enumerate(reranked, start=1):
        candidate["rerank_rank"] = rank

    return reranked

In [62]:
reranked_results = rerank_top10(query, baseline_results)
reranked_top3 = reranked_results[:FINAL_K]

for item in reranked_results:
    print(
        f"New rank: {item['rerank_rank']} | "
        f"Original rank: {item['initial_rank']} | "
        f"Page: {item['page_number']} | "
        f"Cross score: {item['cross_score']:.4f}"
    )
    print(item["text"][:350])
    print("-" * 80)

New rank: 1 | Original rank: 1 | Page: 62 | Cross score: 8.0485
SECTION 3 Obligations of providers and deployers of high-risk AI systems and other parties Article 16 Obligations of providers of high-risk AI systems Providers of high-risk AI systems shall: (a) ensure that their high-risk AI systems are compliant with the requirements set out in Section 2; (b) indicate on the high-risk AI system or, where that is
--------------------------------------------------------------------------------
New rank: 2 | Original rank: 2 | Page: 67 | Cross score: 7.8999
Article 25 Responsibilities along the AI value chain 1. Any distributor, importer, deployer or other third-party shall be considered to be a provider of a high-risk AI system for the purposes of this Regulation and shall be subject to the obligations of the provider under Article 16, in any of the following circumstances: (a) they put their name or
--------------------------------------------------------------------------------
New rank

In [63]:
reranked_by_id = {
    item["chunk_id"]: item for item in reranked_results
}

rank_rows = []

for item in baseline_results:
    updated = reranked_by_id[item["chunk_id"]]

    rank_rows.append({
        "chunk_id": item["chunk_id"],
        "page": item["page_number"],
        "bi_rank": item["initial_rank"],
        "cross_rank": updated["rerank_rank"],
        "rank_change": item["initial_rank"] - updated["rerank_rank"],
        "baseline_top3": item["initial_rank"] <= FINAL_K,
        "reranked_top3": updated["rerank_rank"] <= FINAL_K
    })

rank_df = pd.DataFrame(rank_rows).sort_values("cross_rank")
display(rank_df)

,chunk_id,page,bi_rank,cross_rank,rank_change,baseline_top3,reranked_top3
0,153,62,1,1,0,True,True
1,162,67,2,2,0,True,True
3,66,24,4,3,1,False,True
4,65,24,5,4,1,False,False
8,67,25,9,5,4,False,False
5,56,21,6,6,0,False,False
9,161,66,10,7,3,False,False
6,158,65,7,8,-1,False,False
2,141,56,3,9,-6,True,False
7,149,60,8,10,-2,False,False


In [64]:
baseline_top3 = baseline_results[:FINAL_K]
reranked_top3 = reranked_results[:FINAL_K]

print("BASELINE TOP 3")
for item in baseline_top3:
    print(f"Page {item['page_number']} | Chunk {item['chunk_id']}")
    print(item["text"][:300])
    print("-" * 60)

print("\nRERANKED TOP 3")
for item in reranked_top3:
    print(f"Page {item['page_number']} | Chunk {item['chunk_id']}")
    print(item["text"][:300])
    print("-" * 60)

BASELINE TOP 3
Page 62 | Chunk 153
SECTION 3 Obligations of providers and deployers of high-risk AI systems and other parties Article 16 Obligations of providers of high-risk AI systems Providers of high-risk AI systems shall: (a) ensure that their high-risk AI systems are compliant with the requirements set out in Section 2; (b) ind
------------------------------------------------------------
Page 67 | Chunk 162
Article 25 Responsibilities along the AI value chain 1. Any distributor, importer, deployer or other third-party shall be considered to be a provider of a high-risk AI system for the purposes of this Regulation and shall be subject to the obligations of the provider under Article 16, in any of the f
------------------------------------------------------------
Page 56 | Chunk 141
2. Where a product contains an AI system, to which the requirements of this Regulation as well as requirements of the Union harmonisation legislation listed in Section A of Annex I apply, providers sha

## Section 6 — LLM Answer Generation

### 6.1 Initializing a Local LLM

To avoid dependency on an external API, we use a locally hosted instruction-tuned language model for answer generation. The same model is used for both the baseline and reranking pipelines to ensure a consistent comparison. Only the selected document chunks and the shared prompt are passed to the model.

In [65]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

LOCAL_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL)

model = AutoModelForCausalLM.from_pretrained(
    LOCAL_MODEL,
    torch_dtype=torch.float32,
    device_map="cpu"
)

model.eval()

print("Local LLM loaded successfully.")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Local LLM loaded successfully.


In [66]:
def build_prompt(query, selected_chunks):
    context_parts = []

    for i, item in enumerate(selected_chunks, start=1):
        context_parts.append(
            f"[Context {i} | Page {item['page_number']} | "
            f"Chunk {item['chunk_id']}]\n{item['text']}"
        )

    context = "\n\n".join(context_parts)

    prompt = f"""
Answer the question about the EU Artificial Intelligence Act
using only the document context below.

Instructions:
1. Give a clear and accurate answer.
2. Preserve important conditions, exceptions, and role-specific obligations.
3. Do not invent legal requirements.
4. If the context is insufficient, say so.
5. Cite relevant article numbers and page numbers when identifiable.

Document context:
{context}

Question:
{query}

Answer:
"""
    return prompt

In [67]:
def generate_answer(query, selected_chunks):
    prompt = build_prompt(query, selected_chunks)

    messages = [
        {
            "role": "system",
            "content": (
                "Answer questions using only the supplied document context. "
                "Preserve important conditions and exceptions. "
                "Do not invent unsupported information. "
                "Give a concise, complete answer and finish all sentences."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    )

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=600,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    input_length = inputs["input_ids"].shape[-1]
    new_tokens = outputs[0][input_length:]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

### 6.2 Generating Answers from Both Retrieval Pipelines

The same local language model and prompt template are used to generate answers from both retrieval pipelines. The baseline answer uses the Top 3 chunks selected by the Bi-Encoder, while the reranked answer uses the Top 3 chunks selected by the Cross-Encoder. This keeps the answer-generation model consistent so that differences can be analyzed in relation to the retrieved evidence.

In [68]:
query = "What obligations apply to providers of high-risk AI systems?"
answer1 = generate_answer(query, baseline_top3)
answer2 = generate_answer(query, reranked_top3)

print("ANSWER 1 — BASELINE RETRIEVAL")
print(answer1)

print("\n" + "=" * 80)

print("ANSWER 2 — CROSS-ENCODER RERANKING")
print(answer2)

ANSWER 1 — BASELINE RETRIEVAL
Providers of high-risk AI systems must:

- Ensure compliance with the requirements set out in Section 2
- Indicate their name, registered trade name, or registered trade mark, and contact details on the high-risk AI system or its packaging or accompanying documentation
- Have a quality management system in place that complies with Article 17
- Keep the documentation referred to in Article 18
- Maintain logs automatically generated by their high-risk AI systems as per Article 19
- Undertake the relevant conformity assessment procedure as per Article 43 prior to placing the system on the market or putting it into service
- Draw up an EU declaration of conformity in accordance with Article 47
- Affix the CE marking to the high-risk AI system or its packaging or accompanying documentation if not possible, indicating conformity with this Regulation
- Comply with the registration obligations referred to in Article 49(1)
- Take corrective actions and provide info

In [69]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_DIR / "baseline_answer.txt", "w", encoding="utf-8") as f:
    f.write(answer1)

with open(OUTPUT_DIR / "reranked_answer.txt", "w", encoding="utf-8") as f:
    f.write(answer2)

print("Answers saved to:", OUTPUT_DIR)

Answers saved to: c:\Users\HP\Downloads\LMA_Assignment1\outputs


### 6.4 Comparing Baseline Retrieval and Cross-Encoder Reranking

We compare the initial Bi-Encoder rankings with the Cross-Encoder rankings to determine which passages enter or leave the Top 3. Rank changes help us understand whether reranking prioritizes passages that are more relevant to the query. Changes in ranking alone do not establish an improvement; the retrieved passages must also be examined for relevance and correctness.

In [70]:
import pandas as pd

reranked_by_id = {
    item["chunk_id"]: item
    for item in reranked_results
}

rank_rows = []

for item in baseline_results:
    updated = reranked_by_id[item["chunk_id"]]

    rank_rows.append({
        "chunk_id": item["chunk_id"],
        "page": item["page_number"],
        "bi_rank": item["initial_rank"],
        "cross_rank": updated["rerank_rank"],
        "rank_change": item["initial_rank"] - updated["rerank_rank"],
        "baseline_top3": item["initial_rank"] <= FINAL_K,
        "reranked_top3": updated["rerank_rank"] <= FINAL_K
    })

rank_df = pd.DataFrame(rank_rows).sort_values("cross_rank")
display(rank_df)

,chunk_id,page,bi_rank,cross_rank,rank_change,baseline_top3,reranked_top3
0,153,62,1,1,0,True,True
1,162,67,2,2,0,True,True
3,66,24,4,3,1,False,True
4,65,24,5,4,1,False,False
8,67,25,9,5,4,False,False
5,56,21,6,6,0,False,False
9,161,66,10,7,3,False,False
6,158,65,7,8,-1,False,False
2,141,56,3,9,-6,True,False
7,149,60,8,10,-2,False,False


In [71]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_DIR / "baseline_answer.txt", "w", encoding="utf-8") as f:
    f.write(answer1)

with open(OUTPUT_DIR / "reranked_answer.txt", "w", encoding="utf-8") as f:
    f.write(answer2)

rank_df.to_csv(
    OUTPUT_DIR / "ranking_comparison.csv",
    index=False
)

print("Current answers and ranking comparison saved.")

Current answers and ranking comparison saved.


### 7. Failure Mode Analysis: Stress Testing

We evaluate the retrieval pipelines using queries designed to expose weaknesses in semantic retrieval, including lexical distractors, exceptions, and negation. For each query, the Bi-Encoder retrieves ten candidate passages, and the Cross-Encoder reranks those same candidates. We inspect the Top 3 passages before and after reranking to identify whether relevant evidence is recovered.

In [72]:
stress_queries = {
    "Negation and constraints":
        "Which AI practices are prohibited under the EU AI Act, and what exceptions or conditions apply?",

    "Exceptions and corner cases":
        "Under what conditions can an AI system performing a narrow procedural or preparatory task be excluded from the high-risk classification?",

    "Similar roles / keyword trap":
        "What obligations apply to deployers of high-risk AI systems, rather than providers?"
}

stress_results = {}

for category, stress_query in stress_queries.items():
    baseline = retrieve_top10(stress_query)
    reranked = rerank_top10(stress_query, baseline)

    stress_results[category] = {
        "query": stress_query,
        "baseline": baseline,
        "reranked": reranked
    }

    print("\n" + "=" * 90)
    print("FAILURE TYPE:", category)
    print("QUERY:", stress_query)

    print("\nBASELINE TOP 3")
    for item in baseline[:FINAL_K]:
        print(
            f"Rank {item['initial_rank']} | "
            f"Page {item['page_number']} | "
            f"Chunk {item['chunk_id']}\n"
            f"{item['text'][:700]}\n"
        )

    print("\nRERANKED TOP 3")
    for item in reranked[:FINAL_K]:
        print(
            f"Rank {item['rerank_rank']} | "
            f"Page {item['page_number']} | "
            f"Chunk {item['chunk_id']}\n"
            f"{item['text'][:700]}\n"
        )


FAILURE TYPE: Negation and constraints
QUERY: Which AI practices are prohibited under the EU AI Act, and what exceptions or conditions apply?

BASELINE TOP 3
Rank 1 | Page 45 | Chunk 119
(b) prohibitions of certain AI practices; (c) specific requirements for high-risk AI systems and obligations for operators of such systems; (d) harmonised transparency rules for certain AI systems; (e) harmonised rules for the placing on the market of general-purpose AI models; (f) rules on market monitoring, market surveillance, governance and enforcement; (g) measures to support innovation, with a particular focus on SMEs, including start-ups. Article 2 Scope 1. This Regulation applies to: (a) providers placing on the market or putting into service AI systems or placing on the market general-purpose AI models in the Union, irrespective of whether those providers are established or located 

Rank 2 | Page 51 | Chunk 131
Article 4 AI literacy Providers and deployers of AI systems shall take measures t

In [73]:
# Inspect the relevant chunks for the failure analysis

for chunk_id in [119, 131, 198, 39, 40, 37, 153, 162, 164]:
    matches = [c for c in chunks if c["chunk_id"] == chunk_id]

    if matches:
        c = matches[0]
        print(f"\n{'=' * 80}")
        print(f"Chunk {chunk_id} | Page {c['page_number']}")
        print(c["text"])


Chunk 119 | Page 45
(b) prohibitions of certain AI practices; (c) specific requirements for high-risk AI systems and obligations for operators of such systems; (d) harmonised transparency rules for certain AI systems; (e) harmonised rules for the placing on the market of general-purpose AI models; (f) rules on market monitoring, market surveillance, governance and enforcement; (g) measures to support innovation, with a particular focus on SMEs, including start-ups. Article 2 Scope 1. This Regulation applies to: (a) providers placing on the market or putting into service AI systems or placing on the market general-purpose AI models in the Union, irrespective of whether those providers are established or located within the Union or in a third country; (b) deployers of AI systems that have their place of establishment or are located within the Union; (c) providers and deployers of AI systems that have their place of establishment or are located in a third country, where the output produc

## Latency profiling

In [74]:
import time
import pandas as pd

def profile_query(query, repeats=5):
    # Warm-up to reduce first-run loading effects.
    candidates = retrieve_top10(query)
    rerank_top10(query, candidates)

    baseline_times = []
    reranking_times = []

    for _ in range(repeats):
        start = time.perf_counter()
        candidates = retrieve_top10(query)
        baseline_times.append(time.perf_counter() - start)

        start = time.perf_counter()
        reranked = rerank_top10(query, candidates)
        reranking_times.append(time.perf_counter() - start)

    baseline_mean = sum(baseline_times) / repeats
    reranking_mean = sum(reranking_times) / repeats

    return {
        "Query": query,
        "Baseline retrieval (s)": baseline_mean,
        "Cross-Encoder reranking (s)": reranking_mean,
        "Advanced retrieval total (s)": (
            baseline_mean + reranking_mean
        )
    }

latency_queries = [
    "What obligations apply to providers of high-risk AI systems?",
    "Which AI practices are prohibited under the EU AI Act, "
    "and what exceptions or conditions apply?",
    "What obligations apply to deployers of high-risk AI "
    "systems, rather than providers?"
]

latency_df = pd.DataFrame([profile_query(q) for q in latency_queries])

display(latency_df.round(4))

,Query,Baseline retrieval (s),Cross-Encoder reranking (s),Advanced retrieval total (s)
0,What obligations apply to providers of high-ri...,0.0313,0.4284,0.4597
1,Which AI practices are prohibited under the EU...,0.0319,0.4367,0.4686
2,What obligations apply to deployers of high-ri...,0.0320,0.4264,0.4585


In [75]:
rank_shift_rows = []

for category, result in stress_results.items():
    reranked_by_id = {
        item["chunk_id"]: item
        for item in result["reranked"]
    }

    for item in result["baseline"]:
        updated = reranked_by_id[item["chunk_id"]]

        rank_shift_rows.append({
            "Failure type": category,
            "Chunk ID": item["chunk_id"],
            "Page": item["page_number"],
            "Original rank": item["initial_rank"],
            "Reranked rank": updated["rerank_rank"],
            "Rank improvement": (
                item["initial_rank"] - updated["rerank_rank"]
            ),
            "Entered Top 3": (
                item["initial_rank"] > FINAL_K
                and updated["rerank_rank"] <= FINAL_K
            )
        })

stress_rank_df = pd.DataFrame(rank_shift_rows)

display(
    stress_rank_df.sort_values(
        ["Failure type", "Reranked rank"]
    )
)

,Failure type,Chunk ID,Page,Original rank,Reranked rank,Rank improvement,Entered Top 3
10,Exceptions and corner cases,39,15,1,1,0,False
17,Exceptions and corner cases,37,14,8,2,6,True
11,Exceptions and corner cases,40,15,2,3,-1,False
16,Exceptions and corner cases,111,41,7,4,3,False
13,Exceptions and corner cases,143,57,4,5,-1,False
19,Exceptions and corner cases,145,58,10,6,4,False
14,Exceptions and corner cases,151,61,5,7,-2,False
12,Exceptions and corner cases,149,60,3,8,-5,False
18,Exceptions and corner cases,56,21,9,9,0,False
15,Exceptions and corner cases,152,61,6,10,-4,False


In [76]:
answer_comparisons = {}

for category, result in stress_results.items():
    query = result["query"]

    # Pipeline 1: Original Bi-Encoder Top 3
    baseline_top3 = sorted(
        result["baseline"],
        key=lambda item: item["initial_rank"]
    )[:FINAL_K]

    # Pipeline 2: Cross-Encoder re-ranked Top 3
    reranked_top3 = sorted(
        result["reranked"],
        key=lambda item: item["rerank_rank"]
    )[:FINAL_K]

    print("\n" + "=" * 90)
    print("FAILURE TYPE:", category)
    print("QUERY:", query)

    print("\nGENERATING BASELINE ANSWER...")
    answer1 = generate_answer(query, baseline_top3)

    print("\nGENERATING RERANKED ANSWER...")
    answer2 = generate_answer(query, reranked_top3)

    answer_comparisons[category] = {
        "query": query,
        "answer1": answer1,
        "answer2": answer2,
        "baseline_top3": baseline_top3,
        "reranked_top3": reranked_top3
    }

    print("\n--- ANSWER 1: BASELINE ---\n")
    print(answer1)

    print("\n--- ANSWER 2: RE-RANKED ---\n")
    print(answer2)



FAILURE TYPE: Negation and constraints
QUERY: Which AI practices are prohibited under the EU AI Act, and what exceptions or conditions apply?

GENERATING BASELINE ANSWER...

GENERATING RERANKED ANSWER...

--- ANSWER 1: BASELINE ---

The EU Artificial Intelligence Act prohibits the following AI practices:

1. Placing on the market, putting into service, or use of an AI system that deploys subliminal techniques beyond a person's consciousness or purposefully manipulative or deceptive techniques, with the objective, or the effect of materially distorting the behavior of a person or a group of persons by appreciably impairing their ability to make an informed decision, thereby causing them to take a decision that they would not have otherwise taken in a manner that causes or is reasonably likely to cause that person, another person or group of persons significant harm.

2. Placing on the market, putting into service or use of an AI system that exploits any of the vulnerabilities of a natu

In [77]:
# Inspect the Top 3 passages used for each answer.

for category, result in answer_comparisons.items():
    print("\n" + "=" * 90)
    print("FAILURE TYPE:", category)
    print("QUERY:", result["query"])

    for pipeline, key in [
        ("BASELINE TOP 3", "baseline_top3"),
        ("RERANKED TOP 3", "reranked_top3")
    ]:
        print(f"\n--- {pipeline} ---")

        for item in result[key]:
            print("\nChunk ID:", item.get("chunk_id"))
            print("Page:", item.get("page_number"))
            print("Original rank:", item.get("initial_rank"))
            print("Reranked rank:", item.get("rerank_rank"))

            # Print the passage text stored in the result.
            text_fields = [
                (name, value)
                for name, value in item.items()
                if isinstance(value, str) and len(value) > 150
            ]

            if text_fields:
                for name, value in text_fields:
                    print(f"\n{name}:\n{value}")
            else:
                print("Available fields:", list(item.keys()))



FAILURE TYPE: Negation and constraints
QUERY: Which AI practices are prohibited under the EU AI Act, and what exceptions or conditions apply?

--- BASELINE TOP 3 ---

Chunk ID: 119
Page: 45
Original rank: 1
Reranked rank: None

text:
(b) prohibitions of certain AI practices; (c) specific requirements for high-risk AI systems and obligations for operators of such systems; (d) harmonised transparency rules for certain AI systems; (e) harmonised rules for the placing on the market of general-purpose AI models; (f) rules on market monitoring, market surveillance, governance and enforcement; (g) measures to support innovation, with a particular focus on SMEs, including start-ups. Article 2 Scope 1. This Regulation applies to: (a) providers placing on the market or putting into service AI systems or placing on the market general-purpose AI models in the Union, irrespective of whether those providers are established or located within the Union or in a third country; (b) deployers of AI syste